# SAADE v5 · 04 — Контроль готовності та одноразовий hidden test

**Цей ноутбук призначений лише для `matched_tts` (FINAL), не для `pilot_existing` (PILOT).** Спочатку має бути повністю завершений `02_Full_Dissertation_Experiment.ipynb` у режимі `DATA_MODE='matched_tts'`, а до цього — згенеровано справжні TTS-файли з підтвердженими голосами. Перевірка нижче покаже точні місця результатів і зрозуміло повідомить, чого бракує. Вона **не відкриває** hidden test. Змінні запуску hidden test залишені вимкненими.

Не підміняйте `FINAL` на `PILOT` і не копіюйте `V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv` між експериментами. Папка `Melnyk_dus` не використовується.

## 0. Підключення

In [ ]:
from pathlib import Path
from dataclasses import replace
import sys, json, shutil, importlib.util
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path('/home/jovyan/work/audio_real_fake')
AUA = PROJECT / 'AUA'
RESULTS = PROJECT / 'results'
OUT = RESULTS / 'SAADE_AUA_2026_v5_FINAL'
assert AUA.is_dir(), f'Не знайдено AUA: {AUA}'
assert shutil.which('ffmpeg'), 'Потрібно встановити ffmpeg, щоб відтворити MP3-сценарії'
sys.path.insert(0,str(Path.cwd().resolve()))
import saade_final_core as S
import saade_final_data as D
import saade_final_methods as M
import saade_final_visuals as V
import saade_final_adaptive as A
for mod in (S,D,M,V,A):
    assert Path(mod.__file__).resolve().parent == Path.cwd().resolve(), f'Чужа версія модуля: {mod.__file__}'
cfg=S.Config(root=str(AUA),out=str(OUT),scenario_profile='v2_5level',
             seed=42,seconds=4.,min_seconds=2.,augmentation_probability=.50,
             batch_size=12,lr=.0005,epochs=15,use_cache=True)
print('AUA:',AUA,'\nRESULTS:',OUT)
print('Дані лише для читання. Melnyk_dus не використовується.')

## 1. Перевірка FINAL-каталогу та зафіксованого поділу

Переконайтеся, що основний ноутбук завершився саме в `matched_tts`. Якщо існує лише `PILOT`, четвертий етап ще не готовий.

In [ ]:
from IPython.display import display

SPLITS = OUT / 'V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv'
PILOT_OUT = RESULTS / 'SAADE_AUA_2026_v5_PILOT'
PILOT_SPLITS = PILOT_OUT / 'V5_02_LOCKED_EXPERIMENT_SEGMENTS.csv'

print('Очікується FINAL:', OUT)
print('FINAL split exists:', SPLITS.is_file())
print('PILOT split exists:', PILOT_SPLITS.is_file())
print('FINAL validation predictions:',
      (OUT / 'V5_04_VALIDATION_PREDICTIONS_ALL.csv').is_file())

if not SPLITS.is_file():
    if PILOT_SPLITS.is_file():
        print('ЗНАЙДЕНО ЛИШЕ PILOT. Це пілотний запуск, а не matched REAL–TTS FINAL.')
    else:
        print('Фінальний поділ ще не створений. Запустіть 02_Full_Dissertation_Experiment.ipynb.')
    raise FileNotFoundError(
        f'Немає фінальної вибірки: {SPLITS}\n'
        'Потрібно: 01_Text_to_TTS → підтверджені receipts → '
        '02_Full_Dissertation_Experiment з DATA_MODE="matched_tts" → '
        'навчання E1/E2/E3 та перевірка готовності. '
        'Не копіюйте PILOT split і не відкривайте hidden test.'
    )

splits = pd.read_csv(SPLITS, low_memory=False)
required_cols = {'split','label','group_id','parent_file_id','pair_id','text_key','voice_id'}
missing_cols = required_cols - set(splits.columns)
if missing_cols:
    raise ValueError(f'Некоректна структура фінальної вибірки: {sorted(missing_cols)}')

models = ('spectrogram_cnn','raw_wave_cnn','mfcc_svm')
seeds = (42,123,777)
epochs = 15

# Спочатку показуємо наявність реальних результатів; НЕ вважаємо число seeds доказом навчання.
checkpoint_rows = []
for seed in seeds:
    for model in models:
        for regime in ('static','random','adaptive'):
            path = OUT / 'models' / f'{model}_{regime}_seed{seed}_meta.json'
            checkpoint_rows.append({'seed':seed,'model':model,
                                    'regime':regime,'meta_exists':path.is_file()})
checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table.groupby(['model','regime'])['meta_exists'].agg(['sum','count']).reset_index())

trained_seeds = tuple(
    seed for seed in seeds
    if checkpoint_table[checkpoint_table.seed.eq(seed)]['meta_exists'].all()
)
print('Seeds із усіма meta-файлами:', trained_seeds)
info, status = A.research_readiness(
    splits, cfg, mode='matched_tts', trained_seeds=trained_seeds
)
display(info)
display(pd.DataFrame([status]).T.rename(columns={0:'value'}))
print('Кандидат для фінальної перевірки:', status['candidate_for_strict_review'])

if not status['candidate_for_strict_review']:
    print('STOP: не вмикайте FREEZE_AFTER_REVIEW або RUN_HIDDEN_ONE_TIME.')
else:
    print('Попередні критерії пройдено; додатково перевірте ваги, пороги, '
          'джерела та протокол перед одноразовим hidden test.')

## 2. Зафіксувати ваги, пороги та протокол (тільки після перевірки)

In [ ]:
FREEZE_AFTER_REVIEW = False  # Навмисно вимкнено за замовчуванням
if FREEZE_AFTER_REVIEW:
    if not status['candidate_for_strict_review']:
        raise RuntimeError('STOP: критерії наукової готовності не виконано.')
    path = A.freeze_for_hidden_once(
        splits, cfg, models=models, seeds=seeds,
        epochs=epochs, mode='matched_tts', min_voice_groups=3
    )
    print('Протокол зафіксовано:', path)
else:
    print('Протокол ще не зафіксовано; hidden test закритий.')

## 3. Фінальне одноразове оцінювання (явна згода)

In [ ]:
RUN_HIDDEN_ONE_TIME = False  # Вмикайте тільки після наукового затвердження
if RUN_HIDDEN_ONE_TIME:
    if not status['candidate_for_strict_review']:
        raise RuntimeError('STOP: до hidden test дані не готові.')
    if not (OUT / 'V5_HIDDEN_FROZEN_PROTOCOL.json').is_file():
        raise FileNotFoundError('Спочатку зафіксуйте протокол на етапі 2.')
    summary = A.run_hidden_once(splits, cfg, authorization=True)
    display(summary)
else:
    print('Hidden test не виконувався.')

## Важливо

Результати hidden test не можна використовувати для змін E3, архітектур, порогів, групувань або сегментації з подальшим повторним тестуванням на тій самій вибірці.